In [1]:
! pip install datasets
! pip install torch
! pip install evaluate
!pip install optuna transformers torch

! pip install transformers==4.28.1
! pip install accelerate==0.15.0
! pip install tokenizers==0.13.3
!pip install --upgrade google-auth-oauthlib google-auth-httplib2 google-api-python-client google-auth

In [2]:
import json
import torch
from transformers import BertTokenizer, BertForTokenClassification, Trainer, TrainingArguments
from transformers.modeling_outputs import TokenClassifierOutput
from torch.utils.data import Dataset, random_split
import torch.nn as nn
import torch.nn.functional as F

In [2]:
# Google Drive einbinden
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import os

# Get the current working directory
current_directory = os.getcwd()

# List files in the current directory
files = os.listdir(current_directory)

print("Current Directory:", current_directory)
print("Files in the Directory:", files)


Current Directory: /content
Files in the Directory: ['.config', 'team-04', 'sample_data']


In [4]:
# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Laden der Daten
file_path = '/content/drive/My Drive/Colab_Notebooks/modified_texts_with_interruption.json'
with open(file_path, 'r', encoding='utf-8') as f:
    data = json.load(f)

# Begrenzen Sie die Daten auf die ersten 1000 Einträge
data = data[:1000]

# Extrahieren der Texte und Labels aus dem JSON-Datensatz
texts = []
labels = []

for entry in data:
    text = entry["modified_text"]  # Verwenden Sie den modifizierten Text mit den bereits vorhandenen <interruption> Tokens
    texts.append(text)

    # Erstellen Sie Labels für jedes Token im Text (1 für <interruption> Token, 0 für andere)
    label = []
    tokens = text.split()
    for token in tokens:
        if "<interruption>" in token:
            label.append(1)
        else:
            label.append(0)
    labels.append(label)

# Padding der Labels
max_len = 512
padded_labels = [label + [0] * (max_len - len(label)) if len(label) < max_len else label[:max_len] for label in labels]

# Beispiel zur Veranschaulichung der Label-Generierung
example_texts = texts[:5]  # Nehmen wir die ersten 5 Texte
example_labels = labels[:5]

for i, text in enumerate(example_texts):
    print(f"Text {i+1}: {text}")
    print(f"Labels {i+1}: {example_labels[i]}")

# Beispiel zur Veranschaulichung der gepaddeten Labels
for i, label in enumerate(padded_labels[:5]):
    print(f"Padded Labels {i+1}: {label}")


Text 1: Sehr geehrter Herr Präsident! Sehr geehrte Damen und Herren! Verehrte Bürger! In diesen Coronazeiten – wir haben es in dieser Woche schon öfter gehört – ist wenig normal. Die Bewältigung der Coronakrise hat erhebliche Auswirkungen auf den Bundeshaushalt.
Besonders davon betroffen ist – kein Wunder – der Bereich des Einzelplans 11, Arbeit und Soziales. Der Haushaltsentwurf 2021 für das Bundesministerium für Arbeit und Soziales hat einen Umfang von insgesamt rund 165 Milliarden Euro und ist damit wieder der größte Einzelplan im Bundeshaushalt. Vom Entwurf bis zur Bereinigungssitzung wuchs dieser Haushaltsplan auf knapp 1 Milliarde Euro auf.
Ein Teil dieses Aufwuchses ist nachvollziehbar, da sich einige Zuschüsse und Leistungen an die Herbstprojektion und die Steuerschätzung anlehnen und dadurch zur Bereinigungssitzung angepasst werden. Was mich aber ärgert, sind neue Maßnahmen, die erst zur Bereinigungssitzung auftauchen und für die dann neue Mittel beantragt werden. Im letzten J

In [5]:
# Ein benutzerdefiniertes Dataset erstellen
class InterruptionDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=512):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = self.texts[idx]
        labels = self.labels[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            return_token_type_ids=False,
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        input_ids = encoding['input_ids'].flatten()
        attention_mask = encoding['attention_mask'].flatten()
        labels = torch.tensor(labels[:self.max_len] + [0] * (self.max_len - len(labels)), dtype=torch.long)

        return {
            'input_ids': input_ids,
            'attention_mask': attention_mask,
            'labels': labels
        }

# Tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
dataset = InterruptionDataset(texts, padded_labels, tokenizer, max_len=max_len)

# Aufteilen der Daten in Trainings- und Evaluationsdatenset (80% Training, 20% Evaluation)
train_size = int(0.8 * len(dataset))
val_size = len(dataset) - train_size
train_dataset, eval_dataset = random_split(dataset, [train_size, val_size])


/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.10/dist-packages/huggingface_hub/utils/_token.py:89: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

In [34]:
# Modell mit Dropout und Gewichtsnormierung
class CustomBertForTokenClassification(BertForTokenClassification):
    def __init__(self, config):
        super().__init__(config)
        self.dropout = nn.Dropout(p=0.5)
        self.classifier = nn.utils.weight_norm(nn.Linear(config.hidden_size, config.num_labels))

    def forward(self, input_ids=None, attention_mask=None, token_type_ids=None, position_ids=None, head_mask=None, inputs_embeds=None, labels=None, output_attentions=None, output_hidden_states=None, return_dict=None):
        outputs = self.bert(input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids, position_ids=position_ids, head_mask=head_mask, inputs_embeds=inputs_embeds, output_attentions=output_attentions, output_hidden_states=output_hidden_states, return_dict=return_dict)
        sequence_output = self.dropout(outputs[0])
        logits = self.classifier(sequence_output)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss()
            loss = loss_fct(logits.view(-1, self.num_labels), labels.view(-1))

        if not return_dict:
            output = (logits,) + outputs[2:]
            return ((loss,) + output) if loss is not None else output

        return TokenClassifierOutput(loss=loss, logits=logits, hidden_states=outputs.hidden_states, attentions=outputs.attentions)

model = CustomBertForTokenClassification.from_pretrained('bert-base-uncased', num_labels=2).to(device)


Some weights of the model checkpoint at bert-base-uncased were not used when initializing CustomBertForTokenClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing CustomBertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing CustomBertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Some weights of CustomBertForTokenClassification were not initialized from the model checkpoint at bert-base-uncased and

In [35]:
# TrainingArguments
training_args = TrainingArguments(
    output_dir='./results',
    num_train_epochs=3,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=10,
    evaluation_strategy="epoch",  # Evaluation nur am Ende jeder Epoche
    logging_strategy="epoch",  # Logging nur am Ende jeder Epoche
    save_strategy="epoch",  # Speichern nur am Ende jeder Epoche
    load_best_model_at_end=True,
    fp16=True  # Mixed precision training
)

# Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset
)

# Training
trainer.train()


Epoch,Training Loss,Validation Loss
1,0.016800,0.014983
2,0.014400,0.014012
3,0.013500,0.013242


TrainOutput(global_step=30000, training_loss=0.01488987274169922, metrics={'train_runtime': 16113.0464, 'train_samples_per_second': 14.895, 'train_steps_per_second': 1.862, 'total_flos': 6.27112230912e+16, 'train_loss': 0.01488987274169922, 'epoch': 3.0})

In [ ]:
# Model Saving
model.save_pretrained('./saved_model')
tokenizer.save_pretrained('./saved_model')

# Evaluation
results = trainer.evaluate()
print(results)


In [36]:
# Evaluation
results = trainer.evaluate()
print(results)

{'eval_loss': 0.013242021203041077, 'eval_runtime': 477.264, 'eval_samples_per_second': 41.906, 'eval_steps_per_second': 5.238, 'epoch': 3.0}


In [37]:
# Modell initialisieren (falls noch nicht geschehen)
model = BertForTokenClassification.from_pretrained('./saved_model', num_labels=2).to(device)

# Vorhersagen auf neuen Texten ohne <interruption> Token
test_texts = ["Sehr geehrte Damen und Herren, heute möchte ich über ein wichtiges Thema sprechen. Entschuldigung, darf ich kurz unterbrechen? <interruption> Es geht um die aktuellen wirtschaftlichen Entwicklungen. Die Zahlen zeigen einen positiven Trend, aber wir müssen vorsichtig sein. Ich habe eine Frage zu den genauen Daten, die Sie erwähnt haben. Wir dürfen nicht vergessen, dass viele Menschen noch immer von der Krise betroffen sind. Es ist entscheidend, dass wir weiterhin Maßnahmen zur Unterstützung anbieten. Vielen Dank für Ihre Aufmerksamkeit."]
test_encodings = tokenizer(test_texts, truncation=True, padding=True, max_length=512, return_tensors="pt")
test_encodings = {key: val.to(device) for key, val in test_encodings.items()}  # Daten auf die GPU verschieben
outputs = model(**test_encodings)
predictions = torch.argmax(outputs.logits, dim=-1)

# Anzeigen der Vorhersagen als Tensor
print(predictions.cpu().numpy())

/usr/local/lib/python3.10/dist-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertForTokenClassification: ['cls.predictions.transform.LayerNorm.weight', 'cls.predictions.transform.dense.bias', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of

[[0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 1 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
  0 0 0 0 0 0 0 0 0 0 0 0 0 0]]
